# Radionica 2 — Prvi ML model: koliko vrijedi stan u Tuzli? 🏠

Na prvoj radionici smo **gledali** podatke. Danas ćemo pustiti računar da **uči** iz njih.

Plan:
1. Napravimo "model" bez ML-a — jedno pravilo koje smo sami smislili
2. Napravimo prvi pravi ML model (linearna regresija)
3. Dodajemo sve više informacija o stanovima
4. Isprobamo drugi model (random forest)
5. **Feature engineering** — sami smišljamo nove kolone
6. Leaderboard 🏆

Ćelije označene sa `# TODO` su vaše. Gdje vidite `___`, tu trebate nešto upisati.

## 0. Priprema
Pokrenite ćeliju ispod (Shift + Enter).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

def rmse(stvarno, predvidjeno):
    """Prosječna greška u KM (kvadratna — velike greške se kažnjavaju više)."""
    return np.sqrt(np.mean((np.array(stvarno) - np.array(predvidjeno)) ** 2))

def mae(stvarno, predvidjeno):
    """Prosječna greška u KM (obična — koliko u prosjeku promašimo)."""
    return np.mean(np.abs(np.array(stvarno) - np.array(predvidjeno)))

print("Sve spremno ✅")

## 1. Podaci
Ovo je proširena verzija stanova iz prošle radionice — sada imamo i godinu izgradnje, sprat, lift, parking, opis oglasa...

In [ ]:
df = pd.read_csv("data/stanovi_r2.csv")
print(df.shape)
df.head()

In [ ]:
df.describe()

## 2. "Model" bez ML-a: pravilo koje smo sami napisali

Najjednostavniji način da procijenimo cijenu stana:

> **cijena = kvadratura × 4160 KM**

Ovo je 🟢 *običan program* — mi smo napisali pravilo. Koliko je dobro?

In [ ]:
df["pravilo"] = df["kvadratura_m2"] * 4160

print(f"MAE  (prosječna greška):        {mae(df['cijena_km'], df['pravilo']):,.0f} KM")
print(f"RMSE (kažnjava velike greške):  {rmse(df['cijena_km'], df['pravilo']):,.0f} KM")

**MAE** = u prosjeku promašimo za toliko KM.
**RMSE** = slično, ali velike greške boli više. Danas koristimo RMSE za leaderboard — **manje je bolje**.

In [ ]:
# TODO: Probajte neki drugi broj umjesto 4160. Možete li smanjiti grešku?
cijena_po_m2 = ___

df["pravilo"] = df["kvadratura_m2"] * cijena_po_m2
print(f"RMSE: {rmse(df['cijena_km'], df['pravilo']):,.0f} KM")

💬 Šta mislite — da li postoji *jedan* broj koji dobro radi za stan u Centru i stan u Gornjoj Tuzli?

## 3. Sakrijemo dio stanova (train / test)

Model uči na **80%** stanova (*train*), a provjeravamo ga na **20%** koje nikad nije vidio (*test*).

Kao ispit: ako ste vidjeli tačna pitanja unaprijed, odličan rezultat ne znači da znate gradivo. 📚

In [ ]:
y = df["cijena_km"]   # ono što predviđamo (target)

train_idx, test_idx = train_test_split(df.index, test_size=0.2, random_state=42)
print(f"Train: {len(train_idx)} stanova  |  Test: {len(test_idx)} stanova")

# Koliko je dobro naše pravilo na TEST stanovima?
rmse_pravilo = rmse(y[test_idx], df.loc[test_idx, "kvadratura_m2"] * 4160)
print(f"Pravilo na test skupu: RMSE = {rmse_pravilo:,.0f} KM")

## 4. Prvi ML model 🔵 — linearna regresija

Ista ideja kao pravilo, ali **računar sam pronalazi brojeve**.

Svaki model u scikit-learn radi isto:
- `model.fit(X, y)` → uči
- `model.predict(X)` → predviđa

In [ ]:
X = df[["kvadratura_m2"]]

model = LinearRegression()
model.fit(X.loc[train_idx], y[train_idx])          # učenje
predikcije = model.predict(X.loc[test_idx])         # predviđanje

print(f"Model je sam naučio: cijena ≈ {model.coef_[0]:,.0f} KM × m² + {model.intercept_:,.0f} KM")
print(f"RMSE na test skupu: {rmse(y[test_idx], predikcije):,.0f} KM")

Pogledajmo obje linije na grafikonu. Svaka tačka je jedan stan iz train skupa.
Isprekidane vertikalne crte su **greške** — model bira liniju kod koje su te greške ukupno najmanje.

In [ ]:
x = np.linspace(20, 165, 100)
plt.figure(figsize=(9, 6))
plt.scatter(df.loc[train_idx, "kvadratura_m2"], y[train_idx], alpha=0.25, label="stanovi (train)")
plt.plot(x, x * 4160, "r--", linewidth=2, label="naše pravilo: m² × 4160")
plt.plot(x, model.predict(pd.DataFrame({"kvadratura_m2": x})), "g-", linewidth=2,
         label=f"linearna regresija: {model.coef_[0]:,.0f} × m² + {model.intercept_:,.0f}")

# nekoliko grešaka (udaljenost tačke od zelene linije)
primjer = df.loc[train_idx].sample(8, random_state=1)
for _, r in primjer.iterrows():
    p = model.predict(pd.DataFrame({"kvadratura_m2": [r["kvadratura_m2"]]}))[0]
    plt.plot([r["kvadratura_m2"]] * 2, [r["cijena_km"], p], "k:", linewidth=1)
plt.scatter(primjer["kvadratura_m2"], primjer["cijena_km"], color="black", s=25, zorder=3)

plt.xlabel("Kvadratura (m²)")
plt.ylabel("Cijena (KM)")
plt.title("Pravilo koje smo mi napisali vs. linija koju je model naučio")
plt.legend()
plt.show()

💬 Linije su slične — zato model nije mnogo bolji od pravila. Ima premalo informacija: stan od 60 m² u Centru i u Gornjoj Tuzli za njega je isti. Hajde da mu damo više.

## 5. Pomoćna funkcija `isprobaj`

Da ne pišemo isti kod svaki put: dajete joj **model** i **listu kolona**, ona uči i vrati grešku na test skupu.

In [ ]:
rezultati = []   # ovdje pamtimo sve pokušaje

def isprobaj(model, kolone, opis=""):
    X = pd.get_dummies(df[kolone], dtype=float)     # tekstualne kolone pretvara u brojeve
    model.fit(X.loc[train_idx], y[train_idx])
    pred = model.predict(X.loc[test_idx])
    greska = rmse(y[test_idx], pred)
    ime = type(model).__name__
    rezultati.append({"model": ime, "broj_kolona": len(kolone), "RMSE": round(greska), "opis": opis})
    print(f"{ime:25s} {len(kolone):2d} kolona  →  RMSE = {greska:,.0f} KM")
    return pred

isprobaj(LinearRegression(), ["kvadratura_m2"], "samo m²");

## 6. Više informacija → bolji model?

Dodajte brojčane kolone koje mislite da utiču na cijenu.
Dostupne: `kvadratura_m2`, `broj_soba`, `sprat`, `ukupno_spratova`, `godina_izgradnje`, `lift`, `parking`, `balkon`

In [ ]:
# TODO: dopunite listu kolona
brojcane = ["kvadratura_m2", ___]

pred = isprobaj(LinearRegression(), brojcane, "brojčane kolone")

Pogledajmo kako model griješi — svaka tačka je jedan stan. Što bliže crvenoj liniji, to bolje.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y[test_idx], pred, alpha=0.4)
plt.plot([0, 800000], [0, 800000], "r--")
plt.xlabel("Stvarna cijena (KM)")
plt.ylabel("Predviđena cijena (KM)")
plt.title("Stvarno vs. predviđeno")
plt.show()

---
# ☕ Pauza
---

## 7. Tekstualne kolone: naselje, grijanje, stanje

Model razumije samo brojeve. `pd.get_dummies` pretvara npr. `naselje` u više kolona sa 0/1 — po jednu za svako naselje:

In [ ]:
pd.get_dummies(df[["naselje"]], dtype=int).head()

Funkcija `isprobaj` to već radi umjesto vas — samo dodajte kolone u listu.

In [ ]:
# TODO: dodajte "naselje", "grijanje" i "stanje" na brojčane kolone
sa_kategorijama = brojcane + [___]

isprobaj(LinearRegression(), sa_kategorijama, "+ naselje, grijanje, stanje");

## 8. Drugi model: Random Forest 🌳🌳🌳

Puno malih stabala odlučivanja. Svako postavlja pitanja tipa *"veći od 70 m²?"*, *"u Centru?"*, *"izgrađen poslije 2010?"* — i na kraju **glasaju**.

Isti `fit` / `predict`, samo drugi model.

In [ ]:
# TODO: isprobajte RandomForestRegressor na istim kolonama kao gore
isprobaj(RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1), ___, "random forest");

💬 Koji je bolji? Zašto mislite da je tako?

## 9. Feature engineering 🔧 — sami pravimo nove kolone

Sjetite se brainstorma: neke stvari nisu direktno u podacima, ali ih možemo **izvesti** iz postojećih kolona.

Primjer — starost zgrade:

In [ ]:
df["starost"] = 2026 - df["godina_izgradnje"]
df[["godina_izgradnje", "starost"]].head()

Ideje za nove kolone (napravite bar 2–3):

| Nova kolona | Iz čega? | Zašto bi mogla pomoći? |
|---|---|---|
| `prizemlje` | `sprat == 0` | prizemlje je često jeftinije |
| `zadnji_sprat` | `sprat == ukupno_spratova` | krov, vrućina ljeti... |
| `visoko_bez_lifta` | sprat ≥ 4 **i** nema lift | ko nosi kese na 6. sprat? 😅 |
| `nova_gradnja` / `stara_gradnja` | `starost` | cijena ne raste ravnomjerno sa godinama |
| `hitno` | `opis` sadrži "HITNO" | prodavac žuri → niža cijena |

Savjet: `(uslov).astype(int)` pretvara True/False u 1/0.

In [ ]:
# TODO: napravite nove kolone (dopunite uslove)
df["prizemlje"]        = (df["sprat"] == ___).astype(int)
df["visoko_bez_lifta"] = ((df["sprat"] >= 4) & (df["lift"] == ___)).astype(int)
df["hitno"]            = df["opis"].str.contains("___").astype(int)

# Vaše ideje:
# df["nova_gradnja"] = ...
# df["stara_gradnja"] = ...
# df["zadnji_sprat"] = ...

df[["sprat", "lift", "prizemlje", "visoko_bez_lifta", "hitno"]].head()

In [ ]:
# TODO: dodajte nove kolone u listu i isprobajte OBA modela
nove = [___]
sve_kolone = sa_kategorijama + nove

isprobaj(LinearRegression(), sve_kolone, "+ moje nove kolone")
isprobaj(RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1), sve_kolone, "+ moje nove kolone");

⚠️ **Pažnja — varanje!** Da li bi kolona `cijena_po_m2 = cijena_km / kvadratura_m2` pomogla? Bi — ali ona već sadrži odgovor. 
U stvarnosti ne znamo cijenu stana koji tek procjenjujemo. To se zove **data leakage**.

## 10. Leaderboard 🏆

Svi vaši pokušaji, od najboljeg:

In [ ]:
pd.DataFrame(rezultati).sort_values("RMSE").reset_index(drop=True)

Upišite **najbolji RMSE** i **koje ste nove kolone napravile** u zajedničku tabelu.

Imate još vremena? Probajte:
- još neku novu kolonu (npr. iz `opis` — "pogled", "renoviran")
- izbaciti neku kolonu — da li model postaje gori ili bolji?
- `RandomForestRegressor(n_estimators=500, max_depth=10, ...)` — mijenjaju li postavke rezultat?

## 11. Zadaća 📝

Izaberite **jedno**:

1. **Pravi oglas:** nađite jedan stvarni oglas za stan u Tuzli (OLX), upišite njegove podatke i pogledajte koliko vaš model griješi. Da li je oglas "preskup" ili "povoljan"?
2. **Nova kolona:** smislite i napravite još jednu novu kolonu. Da li se RMSE smanjio? Napišite u 2–3 rečenice zašto mislite da jeste / nije.

Za opciju 1 možete koristiti ovaj kod:

In [ ]:
# Primjer: procjena jednog stana (prilagodite vrijednosti)
moj_stan = {
    "kvadratura_m2": 65, "broj_soba": 3, "sprat": 2, "ukupno_spratova": 6,
    "godina_izgradnje": 1985, "lift": 1, "parking": 0, "balkon": 1,
    "naselje": "Stupine", "grijanje": "centralno", "stanje": "useljiv",
}
# Model treba iste kolone kao pri učenju — zato procjenu radimo kroz isti postupak:
novi = pd.DataFrame([moj_stan])
X_sve = pd.get_dummies(pd.concat([df[sa_kategorijama], novi[sa_kategorijama]]), dtype=float)
model = LinearRegression().fit(X_sve.iloc[:-1].loc[train_idx], y[train_idx])
print(f"Procjena: {model.predict(X_sve.iloc[[-1]])[0]:,.0f} KM")